<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/12_robust_student_t_and_loo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 12 — Robust likelihoods and PSIS-LOO

Notebooks 4 and 5 described each participant's reaction times as Gaussian scatter around their own trajectory. A Gaussian makes large deviations from the trajectory very improbable, so a few unusual observations can pull the fit toward themselves. A **Student-t** likelihood has heavier tails: it expects most observations to lie close to the trajectory and allows an occasional one to lie far from it.

This notebook compares four models: Notebook 4's varying-intercept model and Notebook 5's varying-intercept, varying-slope model, each with a Gaussian and with a Student-t likelihood. It asks two questions: does a Student-t likelihood predict the data better, and do varying slopes? Each change is judged both with and without the other. The comparison uses **leave-one-out cross-validation**, which is new in this course: each observation is predicted by the model fitted without it.

All four models are fitted here from scratch, so this notebook does not depend on anything computed in the earlier notebooks.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

participants = sorted(sleep["Subject"].unique(), key=int)
participant_to_idx = {participant: i for i, participant in enumerate(participants)}
participant_idx = sleep["Subject"].map(participant_to_idx).to_numpy()

assert participant_idx.min() == 0
assert participant_idx.max() == len(participants) - 1
assert np.array_equal(
    np.asarray(participants)[participant_idx],
    sleep["Subject"].to_numpy(),
)

print(f"{len(participants)} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

The plotting helper `plot_loo_participants` is supplied; Question 2.3 explains what it plots. Like the participant plots of the previous notebooks, it draws one panel per participant, with the day of sleep deprivation on the horizontal axis and that participant's observations in black. Each panel is titled with its participant.

In [ ]:
PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_loo_participants(dt):
    """One panel per participant: leave-one-out predictive intervals with that participant's data."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        "posterior": dt["posterior"].to_dataset(),
        "posterior_predictive": reshape(dt["posterior_predictive"].to_dataset()),
        "log_likelihood": reshape(dt["log_likelihood"].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
    })

    pc = azp.plot_loo_interval(
        panels,
        var_names=["y"],
        point_estimate="mean",
        ci_probs=(0.50, 0.90),
        cols=["participant"],
        col_wrap=6,
        labeller=azb.labels.NoVarLabeller(),
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={
            "prediction_markers": {"color": "C1", "marker": "_", "size": 80, "width": 2},
            "observed_markers": {"color": "black", "size": 14},
            "title": True,
            "xlabel": False,
            "ylabel": False,
        },
    )

    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    return pc

## 1. Four models to compare

### 1.1 The four models

In all four models, the reaction time $y_i$ varies around an expected trajectory $\mu_{y,i}$ that belongs to the participant $s[i]$ who produced observation $i$, and participant intercepts are drawn from a population, as in Notebooks 4 and 5:

$$
b_{0,s}
\sim
\operatorname{Normal}(\mu_{b0}, sd_{b0}).
$$

The models differ in two respects, which are varied independently. The **mean structure** either gives all participants one shared slope, or lets each participant have their own:

$$
\mu_{y,i}
=
b_{0,s[i]}
+
b_1\,\mathrm{days}_i
\qquad\text{or}\qquad
\mu_{y,i}
=
b_{0,s[i]}
+
b_{1,s[i]}\,\mathrm{days}_i,
\quad
b_{1,s}
\sim
\operatorname{Normal}(\mu_{b1}, sd_{b1}).
$$

The **likelihood** is either Gaussian or Student-t:

$$
y_i
\sim
\operatorname{Normal}(\mu_{y,i}, sd_y)
\qquad\text{or}\qquad
y_i
\sim
\operatorname{StudentT}(\nu, \mu_{y,i}, sd_y).
$$

| Model | Mean structure | Likelihood |
|---|---|---|
| `gaussian_intercept` | shared slope | Gaussian: Notebook 4's model |
| `gaussian_slope` | varying slopes | Gaussian: Notebook 5's model |
| `student_t_intercept` | shared slope | Student-t |
| `student_t_slope` | varying slopes | Student-t |

As in Notebook 4, the shared slope is `b1`, a single value for all participants. As in Notebook 5, the varying slopes are `b1` with one value per participant, and their population average is `mu_b1`. Every prior except the one for $\nu$ is carried over unchanged from Notebooks 4 and 5.

### 1.2 What does $\nu$ control?

$\nu$ is the Student-t's **degrees of freedom**. Notebook 3 used a Student-t with $\nu = 7$ as a heavy-tailed prior for the slope; here the Student-t describes the observations, and $\nu$ is estimated from the data. How does the Student-t distribution change as $\nu$ grows large, and what does a small $\nu$, such as 2 or 3, say about deviations from the trajectory?

- answer here

### 1.3 Where does the prior for $\nu$ come from, and what does it say?

We use $\nu \sim \operatorname{Gamma}(2, 0.1)$, with shape 2 and rate 0.1. It is the default prior for $\nu$ in the Student-t family of Bambi, the formula interface to PyMC used in the bioassay notebook (checked in Bambi 0.21.0). A default is still a prior choice, so it is worth knowing where it comes from and what it says.

Its mean is 20. It gives about 9% probability to $\nu < 5$, heavy tails, and about 20% to $\nu > 30$, practically Gaussian tails, so it lets the data choose between them. It makes values below 2, with no finite standard deviation, improbable: about 2%.

The other priors were checked in Notebooks 4 and 5. This notebook is about checking predictions after fitting, so it makes no new prior predictive check.

### 1.4 The supplied model-building function

One function builds all four models, with the likelihood and the mean structure as its two arguments. The prior constants are Notebook 5's; Notebook 4 used the same values for its intercepts, its shared slope, and `sd_y`. As in Notebooks 4 and 5, `b0` and `b1` are written in the centered form: each participant's value is drawn directly from the population distribution.

In [ ]:
coords = {
    "obs_id": np.arange(len(sleep)),
    "participant": participants,
}

# Hyperprior constants for the intercept hierarchy (Notebooks 4 and 5)
mu_mu_b0 = 250   # ms
sd_mu_b0 = 100   # ms
sd_sd_b0 = 25    # ms

# Hyperprior constants for the slope (Notebook 5; Notebook 4's shared slope used the first two)
mu_mu_b1 = 0     # ms/day
sd_mu_b1 = 20    # ms/day
sd_sd_b1 = 10    # ms/day

# Prior constant for observation-level variation (Notebooks 4 and 5)
mu_sd_y = 50     # ms

# Prior constants for the Student-t degrees of freedom (Bambi's default)
alpha_nu = 2
beta_nu = 0.1


def build_model(likelihood, varying_slope):
    """Gaussian or Student-t likelihood, with a shared slope or varying slopes."""
    with pm.Model(coords=coords) as model:
        days = pm.Data("days", sleep["Days"].to_numpy(), dims="obs_id")
        pidx = pm.Data("participant_idx", participant_idx, dims="obs_id")

        # Varying intercepts
        mu_b0 = pm.Normal("mu_b0", mu=mu_mu_b0, sigma=sd_mu_b0)
        sd_b0 = pm.Exponential("sd_b0", scale=sd_sd_b0)
        b0 = pm.Normal("b0", mu=mu_b0, sigma=sd_b0, dims="participant")

        if varying_slope:
            # Varying slopes (Notebook 5)
            mu_b1 = pm.Normal("mu_b1", mu=mu_mu_b1, sigma=sd_mu_b1)
            sd_b1 = pm.Exponential("sd_b1", scale=sd_sd_b1)
            b1 = pm.Normal("b1", mu=mu_b1, sigma=sd_b1, dims="participant")
            mu_y = pm.Deterministic("mu_y", b0[pidx] + b1[pidx] * days, dims="obs_id")
        else:
            # One slope shared by all participants (Notebook 4), with the prior of mu_b1
            b1 = pm.Normal("b1", mu=mu_mu_b1, sigma=sd_mu_b1)
            mu_y = pm.Deterministic("mu_y", b0[pidx] + b1 * days, dims="obs_id")

        # Likelihood
        sd_y = pm.Exponential("sd_y", scale=mu_sd_y)
        if likelihood == "gaussian":
            y = pm.Normal(
                "y",
                mu=mu_y,
                sigma=sd_y,
                observed=sleep["Reaction"].to_numpy(),
                dims="obs_id",
            )
        else:
            nu = pm.Gamma("nu", alpha=alpha_nu, beta=beta_nu)
            y = pm.StudentT(
                "y",
                nu=nu,
                mu=mu_y,
                sigma=sd_y,
                observed=sleep["Reaction"].to_numpy(),
                dims="obs_id",
            )
    return model


models = {
    "gaussian_intercept": build_model("gaussian", varying_slope=False),
    "gaussian_slope": build_model("gaussian", varying_slope=True),
    "student_t_intercept": build_model("student_t", varying_slope=False),
    "student_t_slope": build_model("student_t", varying_slope=True),
}

### 1.5 How is a shared-slope model related to the varying-slope model with the same likelihood?

In the shared-slope models, one `b1` applies to every participant. Which parameter of the varying-slope model would have to be zero to give the same mean structure, and which varying-slope parameter would the shared `b1` then correspond to?

- answer here

### 1.6 Fit the four models.

Each model is sampled with the settings of Notebooks 4 and 5. Sampling all four takes about a minute.

In [ ]:
idatas = {}
for name, model in models.items():
    with model:
        idatas[name] = pm.sample(
            draws=1000,
            tune=1500,
            chains=4,
            random_seed=RANDOM_SEED,
        )

### 1.7 Check the diagnostics of all four fits.

The table screens every parameter of each model, including each participant's intercept and slope: it reports the number of divergences and the worst R-hat and ESS values. The summaries below it show each model's population-level parameters: those with a single value per draw, rather than one per participant or observation.

In [ ]:
screen = {}
for name, idata in idatas.items():
    diagnostics = azs.summary(idata, var_names=["~mu_y"], kind="diagnostics", round_to=2)
    screen[name] = {
        "divergences": int(idata["sample_stats"]["diverging"].sum().item()),
        "largest R-hat": diagnostics["r_hat"].max(),
        "smallest bulk ESS": diagnostics["ess_bulk"].min(),
        "smallest tail ESS": diagnostics["ess_tail"].min(),
    }
display(pd.DataFrame.from_dict(screen, orient="index"))

for name, idata in idatas.items():
    posterior = idata["posterior"]
    population = [var for var in posterior.data_vars if set(posterior[var].dims) == {"chain", "draw"}]
    print(name)
    display(azs.summary(
        idata,
        var_names=population,
        ci_prob=0.90,
        ci_kind="hdi",
        round_to=2,
    ))

### 1.8 Plot the traces of the population-level parameters of `student_t_slope`.

It is the only model with all six population-level parameters, including `nu`. Use `azp.plot_trace_dist`.

In [ ]:
# answer here

### 1.9 Do all four fits meet the diagnostic criteria?

Use the criteria established in Notebook 1: no divergences, R-hat close to 1, adequate bulk and tail ESS, and well-mixed traces.

- answer here

## 2. Difficult-to-predict observations

### 2.1 How well does a model predict an observation it has not seen?

A posterior predictive check compares each observation with predictions from a posterior that was fitted to that same observation, so it flatters the model: an unusual observation pulls the fit toward itself and then looks less unusual. **Leave-one-out (LOO) cross-validation** asks a fairer question: fit the model without observation $i$, and ask how well it predicts $y_i$. Doing this exactly would mean refitting each model 144 times.

**PSIS-LOO** approximates all 144 leave-one-out fits from the one fit we already have. Leaving out one of 144 observations usually changes the posterior only slightly, so the draws of the full posterior can be reweighted to stand in for draws from the posterior without observation $i$: draws under which $y_i$ is probable get less weight, because $y_i$ itself pulled the full posterior toward them. This reweighting is called importance sampling, and **Pareto smoothing**, the PS in PSIS, stabilizes the largest weights. When leaving out an observation would change the posterior a lot, the approximation becomes unreliable, and ArviZ warns about it. Section 4 shows how to read that diagnostic.

### 2.2 What does PSIS-LOO need?

It needs the **pointwise log likelihood**, $\log p(y_i \mid \theta)$ for every observation $i$ and every posterior draw $\theta$, which gives the weights, and posterior predictive draws of $y$, which are reweighted into leave-one-out predictions. PyMC does not store the pointwise log likelihood by default. The next cell adds both to each model's `idata`.

In [ ]:
for name, model in models.items():
    with model:
        pm.compute_log_likelihood(idatas[name])
        pm.sample_posterior_predictive(
            idatas[name],
            var_names=["y"],
            extend_inferencedata=True,
            random_seed=RANDOM_SEED,
        )

### 2.3 Plot the Gaussian varying-slope model's leave-one-out predictions.

For each observation, `plot_loo_participants` shows the leave-one-out predictive distribution of reaction time: its mean as a pink tick, and its central 50% and 90% intervals as darker and lighter blue bars, with the observed reaction time as a black point. Unlike the HDIs used elsewhere in the course, these intervals are equal-tailed: ArviZ computes them from reweighted quantiles. The warning printed with the plot is the one described in Question 2.1.

In [ ]:
plot_loo_participants(idatas["gaussian_slope"])
plt.show()

### 2.4 Which observations does the Gaussian model find hardest to predict?

Look for observations far outside their 90% intervals. Name the participant and the day, and say whether each is unusually slow or unusually fast for that participant.

- answer here

### 2.5 Plot the Student-t varying-slope model's leave-one-out predictions.

In [ ]:
plot_loo_participants(idatas["student_t_slope"])
plt.show()

### 2.6 Does the Student-t likelihood change which observations are hard to predict?

Compare the two plots. Are the same observations far outside their intervals? How do the intervals of the other observations differ?

- answer here

## 3. Robust predictive calibration

### 3.1 What does a calibrated model's LOO-PIT look like?

For each observation, the **LOO-PIT** value is the probability, under its leave-one-out predictive distribution, of a reaction time at or below the observed one. An observation at the predictive median has a LOO-PIT of 0.5; one above everything the model predicted has a LOO-PIT near 1. If the predictive distributions are calibrated, the 144 LOO-PIT values are spread uniformly between 0 and 1: about 10% of them below 0.1, about half between 0.25 and 0.75, and so on.

`azp.plot_loo_pit` plots the empirical cumulative distribution function (ECDF) of the LOO-PIT values minus that of a uniform distribution, the **Δ-ECDF**, so a calibrated model stays close to the dashed zero line. The $p$ printed on the plot comes from a test of uniformity: a value below the stated α = 0.01 indicates miscalibration, and the highlighted points are the observations that contribute most to the departure.

The shape of the curve says how the predictions are miscalibrated. If the predictive distributions are too wide, too many LOO-PIT values fall in the middle, and the curve dips below zero before rising above it. If they are too narrow, too many fall near 0 and 1, and the curve rises above zero before dipping below it.

### 3.2 Plot the LOO-PIT of the two varying-slope models.

In [ ]:
for name in ["gaussian_slope", "student_t_slope"]:
    azp.plot_loo_pit(idatas[name], var_names=["y"], visuals={"title": {"text": name}})
    plt.show()

### 3.3 Are the Gaussian model's leave-one-out predictions calibrated?

Use the $p$ value, the shape of the curve, and the highlighted points.

- answer here

### 3.4 Are the Student-t model's leave-one-out predictions calibrated?

- answer here

### 3.5 How does the Student-t likelihood predict both the typical and the unusual days better?

Compare `sd_y` in the two varying-slope models, and `nu` in the Student-t model, using the summaries in Question 1.7 and the prior in Question 1.3.

- answer here

## 4. Predictive comparison

### 4.1 How does PSIS-LOO score a whole model?

Each observation's **log predictive density**, $\log p(y_i \mid y_{-i})$, measures how probable the model fitted without observation $i$ found the value actually observed; higher is better. Summed over all observations, it gives the model's **expected log pointwise predictive density (ELPD)**. An observation far outside its predictive distribution lowers the ELPD a lot, and a predictive distribution that is wider than necessary lowers it a little for every observation.

`azs.loo` returns the ELPD as `elpd`, with its standard error `se`; `p`, the effective number of parameters, which measures how much the model adapts to the data; and each observation's Pareto $k$ as `pareto_k`, together with `good_k`, the threshold below which $k$ is good. An ELPD means little on its own: it is compared between models fitted to the same observations.

### 4.2 Compute PSIS-LOO for each model.

Use `azs.loo` with `var_name="y"` and `pointwise=True`, and store the results in a dictionary `loos` with the model names as keys. Print each model's ELPD, its standard error, and its largest Pareto $k$.

In [ ]:
# answer here

### 4.3 How should Pareto $k$ be read?

Each observation's Pareto $k$ estimates how heavy the tail of its importance weights is, which grows with how far the posterior without that observation lies from the full posterior. Below 0.7, `good_k` here, the observation's PSIS-LOO estimate is reliable. Above it, the estimate is unreliable, and it tends to be too favorable to the model, because the full posterior has been pulled toward the very observation it is supposed to predict without.

A high $k$ does not mean that sampling failed, or that the observation is an error. It means that the observation is **influential**: the fit changes noticeably without it. It should be diagnosed, not deleted. The exact leave-one-out value of a flagged observation can be obtained by refitting the model without it, and a model that is less sensitive to single observations may not produce high $k$ values at all. Finally, $k$ is itself estimated from the posterior draws, so a value near 0.7 can fall on the other side of the threshold in another run.

### 4.4 Plot the Pareto $k$ values of the two varying-slope models, and list the flagged observations.

In each plot, the horizontal line marks 0.7, and the percentages give the share of observations on either side of it. The horizontal axis numbers the observations in the order of the data, so the table below the plots lists, with its participant and day, every observation whose $k$ exceeds 0.7 in any of the four models.

In [ ]:
for name in ["gaussian_slope", "student_t_slope"]:
    pc = azp.plot_khat(loos[name], hline_values=[0.7], visuals={"hlines": {}, "bin_text": {}})
    pc.get_viz("figure").suptitle(name)
    plt.show()

pareto_k = pd.DataFrame({name: loo.pareto_k.values for name, loo in loos.items()})
flagged = (pareto_k > 0.7).any(axis="columns")
sleep[["Subject", "Days", "Reaction"]].join(pareto_k)[flagged].round(2)

### 4.5 Which models have unreliable PSIS-LOO estimates, and for which observations?

Use the printed largest $k$ values, the plots, and the table, and compare the flagged observations with those of Question 2.4. Why might an unusual day be influential under one likelihood but not under the other? In test runs of this notebook with other random seeds, the largest $k$ of `gaussian_slope` ranged from about 0.8 to 1.3, and that of `gaussian_intercept` from about 0.65 to 0.8.

- answer here

### 4.6 What does a model comparison table show, and what does it not?

`azs.compare` ranks the models by ELPD and reports, for each model:

- `elpd_diff`: its ELPD minus that of the best model, which therefore has 0;
- `dse`: the standard error of that difference. It is computed observation by observation, pairing the two models' predictions of the same observation, so it is usually much smaller than the models' separate `se` values suggest;
- `p_worse`: the probability that the model predicts worse than the best one, from a normal approximation;
- `diag_diff`: a warning when the difference is too small, or the data too few, for that normal approximation to be trusted;
- `diag_elpd`: the number of observations with $k$ above the threshold, whose unreliable estimates can bias the comparison in the model's favor;
- `weight`: the **stacking** weights, the weights of the mixture of the models' predictive distributions that would have predicted the left-out observations best.

`azp.plot_compare` draws each model's `elpd_diff` as a point, with a bar extending one `dse` to either side.

Stacking weights are not probabilities that the models are true: a model can get weight 0 because another model makes the same predictions better, and similar weights do not mean that models are equivalent. A rank is not proof either: judge each ELPD difference against its `dse`, and remember that the comparison is about predictions, not about which model is scientifically true.

### 4.7 Compare the four models.

Use `azs.compare` on `loos` with `round_to=2`, store the table as `comparison`, and plot it with `azp.plot_compare`.

In [ ]:
# answer here

### 4.8 Which model predicts best, and does the high Pareto $k$ of `gaussian_slope` put that in doubt?

In test runs, refitting `gaussian_slope` without each of the observations it flags, which gives their exact leave-one-out values, lowered its ELPD by less than 2 in total.

- answer here

### 4.9 Compare every model with the simplest one.

`azs.compare` measures every difference from the best model unless a `reference` model is given. Repeat the comparison with `reference="gaussian_intercept"`. The column `p_worse` then becomes `p_better`, the probability that each model predicts better than the reference.

In [ ]:
# answer here

### 4.10 How much does each change improve prediction, with and without the other?

Between them, the tables in Questions 4.7 and 4.9 contain all four comparisons in which only one thing changes: the likelihood, with a shared slope and with varying slopes, and the mean structure, with each likelihood. Quote each ELPD difference with its `dse`.

- answer here

### 4.11 Why does each change help more when the other is present?

Compare `nu` in the two Student-t models (summaries in Question 1.7). What kind of deviation from the trajectory remains when all participants share one slope, and what kind remains when slopes vary?

- answer here

## 5. What the comparison answers

### 5.1 What prediction task does this comparison evaluate?

Leave-one-out cross-validation left out one row of the data at a time. What remains known about that observation's participant? Does the comparison tell us how well the models would predict a new participant, or a participant's future reaction times?

- answer here

### 5.2 Does the choice of model change the conclusion about sleep deprivation?

Compare the posterior for the population-average daily effect across the four models, using the summaries in Question 1.7. It is `b1` in the shared-slope models and `mu_b1` in the varying-slope models (Question 1.5).

- answer here

### 5.3 What has this notebook shown?

Summarize what the four-way comparison showed, what the Pareto $k$ diagnostics flagged, and what the comparison does and does not establish.

- answer here